In [ ]:
import pandas as pd

historical_df=pd.read_csv('../../data/processed/match_data/historical_fifa_matches.csv')
elo_df=pd.read_csv('../../data/processed/team_data/elo_ratings_cleaned.csv')

matches = historical_df.copy()
elo = elo_df.copy()



matches["date"] = pd.to_datetime(matches["date"])
elo["date"] = pd.to_datetime(elo["date"])

matches = matches.sort_values("date")
elo = elo.sort_values("date")


home_elo = elo.rename(
    columns={
        "team": "home_team",
        "rating": "home_elo",
        "change": "home_change"
    }
)
away_elo = elo.rename(
    columns={
        "team": "away_team",
        "rating": "away_elo",
        "change": "away_change"
    }
)

home_elo = home_elo.sort_values("date")

matches_home = pd.merge_asof(
    matches.sort_values("date"),
    home_elo,
    on="date",
    by="home_team",
    direction="backward"
)
print(matches_home.shape)
print(matches_home[["date","home_team","home_elo"]].head())

away_elo = away_elo.sort_values("date")

matches_with_elo = pd.merge_asof(
    matches_home.sort_values("date"),
    away_elo,
    on="date",
    by="away_team",
    direction="backward"
)

matches_with_elo["elo_diff"] = (
    matches_with_elo["home_elo"]
    - matches_with_elo["away_elo"]
)

matches_with_elo["change_diff"] = (
    matches_with_elo["home_change"]
    - matches_with_elo["away_change"]
)

print(matches_with_elo[
    ["home_elo", "away_elo"]
].isnull().sum())

print((matches["date"] < elo["date"].min()).sum())

missing_home = matches_with_elo[
    matches_with_elo["home_elo"].isna()
]

print(missing_home["home_team"].value_counts().head(20))

clean_matches = matches_with_elo.dropna(
    subset=["home_elo", "away_elo"]
)

print(len(clean_matches))
matches_with_elo = matches_with_elo.dropna(
    subset=["home_elo", "away_elo"]
)
matches_with_elo.to_csv(
    "../../data/processed/match_data/matches_with_elo.csv",
    index=False
)